# Índice FIBRAS de AMEFIBRA

Notebook para extraer la tabla pública del Índice FIBRAS. La página carga los datos dentro de un `iframe` mediante JavaScript y WebSocket, por lo que se utiliza Playwright con Chromium.

> La información se ofrece únicamente para consulta y análisis. AMEFIBRA indica que los datos tienen aproximadamente 20 minutos de retraso y no deben usarse como base única para decisiones de inversión.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

try:
    # VS Code inyecta esta variable con la ruta absoluta del propio notebook,
    # así que la raíz del proyecto queda anclada a dónde vive el archivo .ipynb,
    # sin importar cuál sea el directorio de trabajo con el que arrancó el kernel
    # (que puede no ser la raíz del proyecto, según la configuración del editor).
    RAIZ_PROYECTO = Path(__vsc_ipynb_file__).resolve().parent
except NameError:
    RAIZ_PROYECTO = Path.cwd()
if str(RAIZ_PROYECTO) not in sys.path:
    sys.path.insert(0, str(RAIZ_PROYECTO))

from modules.presentacion import (
    aplicar_tema_oscuro_notebook,
    armar_tabla_tickers_seleccionados,
    descargar_historiales_dividendos,
    ejecutar_extraccion_indice,
    exportar_csv_excel,
    exportar_xlsx,
    mostrar_comparativo_completo_cliente,
    mostrar_comparativo_rendimiento,
    mostrar_emisoras,
    resumen_precio_periodicidad,
    seleccionar_anio_interactivo,
    seleccionar_tickers_interactivo,
)
from modules.procesamiento import calcular_ventana_movil_12_meses, obtener_anios_disponibles_comunes

In [2]:
# Aplicar tema oscuro al notebook
aplicar_tema_oscuro_notebook()

# Configuración de rutas de salida
CARPETA_SALIDA = Path.cwd() / "output"
CARPETA_FICHAS_EXPORT = CARPETA_SALIDA / "fichas"

# Parámetros editables del notebook
HEADLESS = True
TIMEOUT_DATOS_MS = 30000
EXPORTAR_CSV_ANALITICO = True
EXPORTAR_CSV_EXCEL = False
EXPORTAR_XLSX = False
RUTA_CSV_EXCEL = Path.cwd() / "indice_fibras.csv"
RUTA_XLSX = Path.cwd() / "indice_fibras.xlsx"

## Ejecutar extracción
Consultar solo los lunes temprano para hacer un análisis rápido de las FIBRAS y para saber si hubo altas y bajas de emisoras.

In [ ]:
df = ejecutar_extraccion_indice(HEADLESS, TIMEOUT_DATOS_MS, CARPETA_SALIDA, EXPORTAR_CSV_ANALITICO)

### Exportar a archivo de Excel - xlsx (Ejecución opcional)

In [ ]:
if EXPORTAR_CSV_EXCEL:
    ruta_csv_excel = exportar_csv_excel(df, RUTA_CSV_EXCEL)
    print(f"CSV compatible con Excel guardado en: {ruta_csv_excel}")

if EXPORTAR_XLSX:
    ruta_xlsx = exportar_xlsx(df, RUTA_XLSX)
    print(f"Excel guardado en: {ruta_xlsx}")

## Consulta de emisoras

In [3]:
try:
    df
except NameError:
    df = None

df_emisoras = mostrar_emisoras(df, CARPETA_SALIDA)

Fuente de emisoras: histórico de 20260904_185225_list_of_tickers.csv (no se ejecutó la extracción de AMEFIBRA en esta corrida).
      Emisora
0    DANHOS13
1     EDUCA18
2   FIBRAMQ12
3   FIBRAPL14
4   FIBRAUP18
5      FIHO12
6      FINN13
7      FMTY14
8     FNOVA17
9     FPLUS16
10    FSHOP13
11     FUNO11
12     NEXT25
13     SOMA21
14  STORAGE18


## Historial de distribuciones por FIBRA

### Fuentes evaluadas

| Fuente | Cobertura BMV | Datos de distribuciones | Acceso y límites |
|---|---|---|---|
| Relación con Inversionistas del emisor | Sí, por emisora | Fuente primaria; puede incluir fechas, importe y componentes fiscales en PDF/XLSX | Gratuita, sin API uniforme; requiere localizar y procesar reportes de cada emisor |
| AMEFIBRA | Sí, índice agregado | Cotización e indicadores del índice; no publica aquí un histórico normalizado de distribuciones | Consulta web pública; no se expone una API de dividendos en esta tabla |
| BMV/BIVA | Sí | Información oficial de emisoras y eventos, según disponibilidad del portal | Consulta pública, pero sin una API gratuita y estable para este flujo |
| FMP, Alpha Vantage, Twelve Data, EODHD, Nasdaq Data Link y Polygon | Cobertura mexicana variable | La cobertura y profundidad de dividendos para tickers BMV no está garantizada en el plan gratuito | Requieren revisar ticker, API key y límites por proveedor |
| `yfinance` | Sí para tickers Yahoo con sufijo `.MX`, cuando Yahoo dispone del evento | Fecha ex-dividendo y monto; no garantiza fecha de registro, pago ni componentes fiscales | Gratis y sin API key, pero es un cliente no oficial de Yahoo Finance y está sujeto a cambios y límites |

Se usa `yfinance` como respaldo reproducible porque las fuentes primarias no ofrecen una API homogénea. El resultado contiene la fecha ex-dividendo y el importe disponible en Yahoo; la fecha de registro, fecha de pago y componentes fiscales no se incluyen porque esta fuente no los entrega de forma confiable. El histórico se ordena del más antiguo al más reciente. `yield_pct` es el rendimiento de cada distribución respecto al cierre de su fecha ex-dividendo; `annualized_yield_pct` anualiza ese rendimiento usando `365 / días_del_periodo`. Para la primera fila se usa la mediana histórica de días entre distribuciones.

### Tickers a consultar

Marca una o más FIBRAs en la lista de casillas (mismo listado de la sección "Consulta de emisoras"): un clic por cada ticker que quieras incluir. Al correr esta celda se despliega la lista; ajusta las casillas y luego corre la celda de abajo para consultar los tickers elegidos.

In [4]:
selector_tickers = seleccionar_tickers_interactivo(df_emisoras["Emisora"])

_SelectorTickersMultiple(children=(HTML(value='<b>Tickers a analizar</b> (marca una o varias):'), Checkbox(val…

In [5]:
# Armamos el DataFrame de tickers seleccionados (con la cotización de AMEFIBRA de esta
# corrida como metadato, si se ejecutó la extracción). Sobre él descargamos el historial
# de dividendos de cada ticker y mostramos precio actual y periodicidad en una sola tabla.
try:
    df
except NameError:
    df = None

tickers_seleccionados = armar_tabla_tickers_seleccionados(selector_tickers.value, df_emisoras["Emisora"], df)

historiales = descargar_historiales_dividendos(tickers_seleccionados["ticker"], df_emisoras["Emisora"], CARPETA_SALIDA)
resumen_tickers = resumen_precio_periodicidad(tickers_seleccionados, historiales)

Tickers seleccionados (5): DANHOS13, FIBRAPL14, FMTY14, FNOVA17, FUNO11.
  DANHOS13: 48 distribuciones · periodicidad trimestral · CSV: 20260904_185627_DANHOS13_dividendos.csv
  FIBRAPL14: 51 distribuciones · periodicidad trimestral · CSV: 20260904_185634_FIBRAPL14_dividendos.csv
  FMTY14: 83 distribuciones · periodicidad mensual · CSV: 20260904_185638_FMTY14_dividendos.csv
  FNOVA17: 32 distribuciones · periodicidad trimestral · CSV: 20260904_185639_FNOVA17_dividendos.csv
  FUNO11: 64 distribuciones · periodicidad trimestral · CSV: 20260904_185644_FUNO11_dividendos.csv


,ticker,precio_actual,periodicidad
0,DANHOS13,$29.95,trimestral
1,FIBRAPL14,$75.06,trimestral
2,FMTY14,$14.17,mensual
3,FNOVA17,$41.78,trimestral
4,FUNO11,$29.62,trimestral


## FICHA DE RENDIMIENTO ANUAL PERSONALIZADO

La ficha usa el **año calendario** (`1 de enero` a `31 de diciembre`). Los pagos se filtran por `ex_date`, que es la fecha disponible en el historial de `yfinance`; no se inventa una fecha de pago que la fuente no proporciona. Los precios inicial y final son el primer y último cierre disponible dentro del año. El rendimiento por dividendos se calcula contra el precio inicial, y el rendimiento de capital contra la variación entre precio final e inicial. La ficha es informativa y no constituye una recomendación de inversión.

### Año a consultar

Elige, del desplegable, el año a consultar. Solo se muestran los años con distribuciones disponibles para **todos** los tickers seleccionados (intersección); el año elegido se usa para el comparativo de todas las FIBRAs seleccionadas.

In [6]:
AÑOS_DISPONIBLES = obtener_anios_disponibles_comunes(historiales)
selector_anio = seleccionar_anio_interactivo(AÑOS_DISPONIBLES)

Hay información disponible de 2019 a 2026.


Dropdown(description='Año:', options=(2026, 2025, 2024, 2023, 2022, 2021, 2020, 2019), value=2026)

In [7]:
# Generamos y exportamos (HTML responsivo) el comparativo de rendimiento anual de
# todas las FIBRAs seleccionadas, para el año elegido arriba.
AÑO_SELECCIONADO = selector_anio.value
ruta_comparativo_rendimiento_anual = mostrar_comparativo_rendimiento(
    tickers_seleccionados, historiales, CARPETA_SALIDA, CARPETA_FICHAS_EXPORT, año=AÑO_SELECCIONADO
)

Comparativo de rendimiento generado: d:\devs\dev-workbench\own-projects\tool-python-extract-amefibra-data-fibras\output\20260904_185827_comparativo_2025_rendimiento.html


HTML exportado: d:\devs\dev-workbench\own-projects\tool-python-extract-amefibra-data-fibras\output\fichas\2026-09-04_1858_comparativo_rendimiento-anual_2025.html


### Comparativo de FIBRAs del año seleccionado (ficha completa para cliente)

Comparativo de FIBRAs pensado como entregable final para el cliente (escenario de inversión, distribuciones mensuales y rendimiento total en el año), con un diseño distinto al del comparativo de rendimiento anterior. Junta en una sola tabla a todas las FIBRAs seleccionadas, con el año ya elegido arriba y los datos reales de cada una; no inventa cifras. Es informativa y no constituye una recomendación de inversión.

In [8]:
# Generamos y exportamos (HTML responsivo) el "Comparativo de FIBRAs" (ficha completa
# anual para el cliente) de todas las FIBRAs seleccionadas, con el mismo año.
ruta_comparativo_completo_anual = mostrar_comparativo_completo_cliente(
    tickers_seleccionados, historiales, CARPETA_SALIDA, CARPETA_FICHAS_EXPORT, año=AÑO_SELECCIONADO
)

Comparativo de FIBRAs generado: d:\devs\dev-workbench\own-projects\tool-python-extract-amefibra-data-fibras\output\20260904_185856_comparativo_2025_ficha_completa_cliente.html


HTML exportado: d:\devs\dev-workbench\own-projects\tool-python-extract-amefibra-data-fibras\output\fichas\2026-09-04_1858_comparativo_ficha-completa-cliente_2025.html


## FICHA DE RENDIMIENTO Y RIESGO DE LOS ÚLTIMOS 12 MESES

Misma ficha de rendimiento de arriba, pero calculada sobre la ventana móvil de los últimos 12 meses completos (en vez de año calendario), con el riesgo mensual promedio del periodo (volatilidad del retorno total mensual: variación de precio + dividendos del mes) agregado como cifra destacada junto al rendimiento total.

In [ ]:
# Fecha de referencia para la ventana móvil de 12 meses (fecha_fin del periodo).
# None = usa la fecha actual; fijar una fecha (ej. "2025-12-31") permite correr el
# análisis de forma retrospectiva, útil para pruebas. La ventana es única y se aplica
# por igual a todos los tickers seleccionados. El flujo de año calendario de las celdas
# anteriores no se modifica y sigue disponible como antes.
FECHA_REFERENCIA_12M = None
FECHA_INICIO_12M, FECHA_FIN_12M = calcular_ventana_movil_12_meses(FECHA_REFERENCIA_12M)
print(f"Ventana de análisis (única para todos los tickers seleccionados): {FECHA_INICIO_12M:%Y-%m-%d} a {FECHA_FIN_12M:%Y-%m-%d}")

### Comparativo de rendimiento de los últimos 12 meses

In [10]:
# Generamos y exportamos (HTML responsivo) el comparativo de rendimiento de los
# últimos 12 meses de todas las FIBRAs seleccionadas, con la ventana definida arriba.
ruta_comparativo_rendimiento_12m = mostrar_comparativo_rendimiento(
    tickers_seleccionados, historiales, CARPETA_SALIDA, CARPETA_FICHAS_EXPORT, fecha_referencia=FECHA_REFERENCIA_12M
)

Comparativo de rendimiento generado: d:\devs\dev-workbench\own-projects\tool-python-extract-amefibra-data-fibras\output\20260904_185921_comparativo_20260904_ult12m_rendimiento.html


Indicador,DANHOS13,FIBRAPL14,FMTY14,FNOVA17,FUNO11
Periodo (cierres),2025-09-05 a 2026-09-04,2025-09-05 a 2026-09-04,2025-09-05 a 2026-09-04,2025-09-05 a 2026-09-04,2025-09-05 a 2026-09-04
Precio inicial,$25.83 MXN,$71.39 MXN,$13.45 MXN,$28.15 MXN,$28.51 MXN
Precio final,$29.95 MXN,$75.06 MXN,$14.17 MXN,$41.78 MXN,$29.62 MXN
Variación de precio,15.95%,5.14%,5.35%,48.42%,3.89%
Rendimiento por dividendos,6.28%,3.96%,7.36%,8.75%,8.89%
Rendimiento total,22.23%,9.10%,12.72%,57.17%,12.78%
Ganancia total,$5.74 MXN,$6.50 MXN,$1.71 MXN,$16.09 MXN,$3.64 MXN
Dividendos recibidos,$1.6231 MXN,$2.8256 MXN,$0.9905 MXN,$2.4644 MXN,$2.5348 MXN
Variación de capital,$4.12 MXN,$3.67 MXN,$0.72 MXN,$13.63 MXN,$1.11 MXN
Riesgo mensual promedio,3.10%,5.35%,4.04%,13.79%,4.01%


HTML exportado: d:\devs\dev-workbench\own-projects\tool-python-extract-amefibra-data-fibras\output\fichas\2026-09-04_1859_comparativo_rendimiento-12-meses_20260904.html


### Comparativo de FIBRAs de los últimos 12 meses (ficha completa para cliente)

Mismo comparativo de arriba, pero sobre la ventana móvil de últimos 12 meses: agrega una tabla de riesgo del periodo con la volatilidad anualizada y mensual promedio, y el retorno total mensual de cada FIBRA (verde = mes positivo, rojo = mes negativo), junto al desglose de rendimiento (plusvalía vs. distribuciones).

In [11]:
# Generamos y exportamos (HTML responsivo) el "Comparativo de FIBRAs" (ficha completa
# de los últimos 12 meses) para el cliente de todas las FIBRAs seleccionadas.
ruta_comparativo_completo_12m = mostrar_comparativo_completo_cliente(
    tickers_seleccionados, historiales, CARPETA_SALIDA, CARPETA_FICHAS_EXPORT, fecha_referencia=FECHA_REFERENCIA_12M
)

Comparativo de FIBRAs generado: d:\devs\dev-workbench\own-projects\tool-python-extract-amefibra-data-fibras\output\20260904_185951_comparativo_20260904_ult12m_ficha_completa_cliente.html


Indicador,DANHOS13,FIBRAPL14,FMTY14,FNOVA17,FUNO11
Precio de compra,$25.83,$71.39,$13.45,$28.15,$28.51
Precio actual,$29.95,$75.06,$14.17,$41.78,$29.62
Títulos,387,140,743,355,350
Plusvalía,"$1,594.44",$513.80,$534.96,"$4,838.65",$388.50
Dividendo por título,$1.6231,$2.8256,$0.9905,$2.4644,$2.5348
Distribuciones totales,$628.13,$395.58,$735.93,$874.85,$887.17
Retorno total,"$2,222.57",$909.38,"$1,270.89","$5,713.50","$1,275.67"
Rendimiento total,22.23%,9.09%,12.71%,57.14%,12.76%
Volatilidad mensual promedio,3.10%,5.35%,4.04%,13.79%,4.01%
Volatilidad anualizada,10.75%,18.53%,14.01%,47.76%,13.89%


HTML exportado: d:\devs\dev-workbench\own-projects\tool-python-extract-amefibra-data-fibras\output\fichas\2026-09-04_1859_comparativo_ficha-completa-12-meses_20260904.html


## ANÁLISIS DE TRES AÑOS